## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

from resnet_20_32_44_56_SubExperiment3_convolution import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

net = ResNet20()
# net = ResNet32()
# net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 352ms | Tot: 22s481ms | Loss: nan | Acc: 0.330% (165/50000) 391/391 c: 11.719% (165/1408) 11/391 
  Step: 13ms | Tot: 1s361ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 1
  Step: 56ms | Tot: 22s601ms | Loss: nan | Acc: 0.000% (0/50000) 391/391  
  Step: 13ms | Tot: 1s385ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 2
  Step: 55ms | Tot: 22s152ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 13ms | Tot: 1s347ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 3
  Step: 56ms | Tot: 21s979ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 13ms | Tot: 1s371ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 4
  Step: 57ms | Tot: 22s86ms | Loss: nan | Acc: 0.000% (0/50000) 391/391  
  Step: 13ms | Tot: 1s367ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 5
  Step: 57ms | Tot: 22s35ms | Loss: nan | Acc: 0.000% (0/50000) 391/391  
  Step: 13ms | Tot: 1s363ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 6
  Step: 57ms | 

  Step: 57ms | Tot: 22s456ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 13ms | Tot: 1s377ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 52
  Step: 56ms | Tot: 22s181ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 12ms | Tot: 1s367ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 53
  Step: 57ms | Tot: 22s260ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 13ms | Tot: 1s375ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 54
  Step: 56ms | Tot: 22s213ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 13ms | Tot: 1s369ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 

Epoch: 55
  Step: 56ms | Tot: 22s176ms | Loss: nan | Acc: 0.000% (0/50000) 391/391 
  Step: 12ms | Tot: 1s391ms | Loss: nan | Acc: 0.000% (0/10000) 100/100 


KeyboardInterrupt: 

In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 0
Error: 100
